# Clase 13 · Notebook 02
# Validación cruzada (k-fold) y análisis de residuos

**Introducción al Deep Learning — Módulo 2, Unidad 3: Regresión (Parte II)**

Dos herramientas para confiar en un modelo de regresión:

1. **Validación cruzada (k-fold)**: una estimación robusta del error.
2. **Análisis de residuos**: comprobar si el modelo es válido.

Usamos el dataset *diabetes*.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Validación cruzada k-fold (a mano)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
tf.random.set_seed(42); np.random.seed(42)

d = load_diabetes()
X, y = d.data.astype("float32"), d.target.astype("float32")

def crear():
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(10,)), Dense(32, activation="relu"), Dense(1)])
    m.compile(optimizer="adam", loss="mse"); return m

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2s = []
for i, (tr, te) in enumerate(kf.split(X), 1):
    sc = StandardScaler().fit(X[tr])              # escalar dentro de cada fold
    m = crear()
    m.fit(sc.transform(X[tr]), y[tr], epochs=80, batch_size=16, verbose=0)
    r2 = r2_score(y[te], m.predict(sc.transform(X[te]), verbose=0).ravel())
    r2s.append(r2)
    print(f"fold {i}: R2 = {r2:.3f}")

print(f"\nR2 medio (5-fold): {np.mean(r2s):.3f}  ±  {np.std(r2s):.3f}")

La media de los 5 folds es una estimación **mucho más fiable** del rendimiento que una sola partición,
y la desviación nos dice cómo de estable es el modelo.

## 2. Análisis de residuos

Entrenamos un modelo final y miramos los **residuos** (real − predicho). Si el modelo es adecuado, deben
repartirse al azar alrededor de cero, sin patrón.

In [ ]:
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler().fit(Xtr)
modelo = crear()
modelo.fit(sc.transform(Xtr), ytr, epochs=120, batch_size=16, verbose=0)

pred = modelo.predict(sc.transform(Xte), verbose=0).ravel()
residuos = yte - pred

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(pred, residuos, color="#4355FF", alpha=0.6)
ax[0].axhline(0, color="#FF647E", lw=2)
ax[0].set_xlabel("Predicción"); ax[0].set_ylabel("Residuo (real - predicho)")
ax[0].set_title("Residuos vs. predicción"); ax[0].grid(alpha=0.3)
ax[1].hist(residuos, bins=20, color="#000F9F", alpha=0.7)
ax[1].set_title("Distribución de los residuos"); ax[1].set_xlabel("Residuo")
plt.tight_layout(); plt.show()

print("Media de los residuos: %.2f (idealmente cerca de 0)" % residuos.mean())

- **Residuos vs. predicción**: deben formar una nube sin forma alrededor de la línea cero. Un patrón
  (curva o embudo) indicaría no linealidad o varianza no constante.
- **Histograma**: idealmente centrado en 0 y aproximadamente simétrico.

## Experimenta tú

1. Cambia `n_splits` a 10 y observa la media y la desviación del R².
2. Aumenta el tamaño de la red o las épocas y mira los residuos.
3. Compara el R² medio de k-fold con el de una sola partición train/test.

## Resumen

- La **validación cruzada k-fold** estima el error de forma robusta (media ± desviación de los folds).
- El **análisis de residuos** comprueba que el modelo es válido: residuos al azar y centrados en cero.
- Escala los datos **dentro de cada fold** para no filtrar información.

Con esto cerramos la segunda parte de la unidad de regresión.